# Milestone 6F: Multi-Source Distributed ETL Pipeline with Data Quality Validation

This notebook runs the end-to-end multi-source ETL pipeline, demonstrating:
1. **Multi-Source Extraction**: Ingesting banking transactions, OFAC sanctions lists, and FATF country risk indices.
2. **Schema Mapping & Standardization**: Unifying schemas, currencies (15 currencies to USD), and timestamps.
3. **Deterministic Validation**: Isolating records with null fields, negative amounts, self-transfers, or future timestamps.
4. **Consistency Verification**: Deduplication and sanctions list cross-referencing.
5. **Feature Enrichment**: Attaching jurisdiction risk ratings and transaction velocity indicators.


In [1]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('../src'))

from src.etl_pipeline import create_spark_session, ETLPipeline

spark = create_spark_session('Databricks-M6F-ETL')

tx_path = '../data/processed/processed_df.csv'
if not os.path.exists(tx_path):
    tx_path = '../data/raw/HI-Small_Trans.csv'

sanctions_path = '../data/external/sanctions_list.csv'
country_path = '../data/external/country_risk_scores.csv'

pipeline = ETLPipeline(
    spark=spark,
    transactions_path=tx_path,
    sanctions_path=sanctions_path,
    country_risk_path=country_path,
    max_transaction_rows=50000,
)

result = pipeline.run()
report = pipeline.generate_quality_report()


C:\Projects\Mule-tiplier Extended\chain-reaction-env\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


2026-10-29 12:08:03,285 [INFO] src.etl_pipeline — Starting Multi-Source ETL Pipeline execution...


2026-10-29 12:08:07,356 [INFO] src.etl_pipeline — Extracted: 50000 transactions, 189 sanctions records, 50 country risk profiles


2026-10-29 12:08:12,184 [INFO] src.etl_pipeline — Validation Complete: 13083 valid, 36917 rejected


C:\Projects\Mule-tiplier Extended\chain-reaction-env\Lib\site-packages\pyspark\sql\classic\column.py:361: FutureWarning: A column as 'key' in getItem is deprecated as of Spark 3.0, and will not be supported in the future release. Use `column[key]` or `column.key` syntax instead.
  warnings.warn(


2026-10-29 12:08:17,858 [INFO] src.etl_pipeline — ETL Pipeline completed successfully.


## 1. Data Quality and Audit Report
Inspect the automated audit metrics generated by the ETL pipeline.


In [2]:
print('=== DATA QUALITY AUDIT REPORT ===')
print('Ingestion Counts:', report['total_records_by_source'])
print('Rejection Breakdown:', report['records_rejected_by_rule'])
print('Duplicates Removed:', report['deduplication_removed'])
print('Cleaned Valid Transactions:', report['total_valid_records'])
print('Sanctions Matches:', report['sanctions_matches_found'])
print('Referential Integrity:', report['referential_integrity'])


=== DATA QUALITY AUDIT REPORT ===
Ingestion Counts: {'transactions': 50000, 'sanctions_watchlists': 189, 'country_risk_ratings': 50}
Rejection Breakdown: {'SELF_TRANSFER': 36917}
Duplicates Removed: 0
Cleaned Valid Transactions: 13083
Sanctions Matches: 2
Referential Integrity: {'orphan_senders_count': 0, 'orphan_receivers_count': 0}


## 2. Enriched Account Profiles
Review final enriched profiles containing sanctions tags and jurisdiction risk scores.


In [3]:
result.enriched_accounts.select(
    'account', 'in_degree', 'out_degree', 'total_in', 'is_sanctioned',
    'jurisdiction_code', 'jurisdiction_risk_score', 'max_txns_per_hour', 'is_laundering'
).show(10, truncate=False)
spark.stop()


+---------+---------+----------+--------+-------------+-----------------+-----------------------+-----------------+-------------+
|account  |in_degree|out_degree|total_in|is_sanctioned|jurisdiction_code|jurisdiction_risk_score|max_txns_per_hour|is_laundering|
+---------+---------+----------+--------+-------------+-----------------+-----------------------+-----------------+-------------+
|100428660|1        |944       |0.01    |0            |SE               |0.14                   |944              |1            |
|800042CB0|0        |2         |0.0     |0            |TH               |0.55                   |2                |0            |
|800043210|0        |3         |0.0     |0            |MT               |0.46                   |3                |0            |
|800043C00|0        |4         |0.0     |0            |KP               |0.99                   |4                |0            |
|800054B70|0        |2         |0.0     |0            |ES               |0.34             